# Fermionic Permutation on 2D Grid: Three-Method Comparison

This notebook implements and compares three methods for the **fermionic permutation operator** $\mathcal{F}_\pi = \hat{P}_\pi \hat{V}_\pi$ on an $L \times L$ qubit grid:

| Method | Routing | Phase Correction | CNOT Depth | Ancillas |
|--------|---------|------------------|------------|----------|
| **Baseline** | 1D snake odd-even sort | Built into FSWAP | $\sim 2L^2$ | 0 |
| **Ancilla $\Gamma$** | Hall Row-Col-Row | $\Gamma$ with ancillas (Jiang et al.) | $\sim 20L$ | $L$ |
| **Ancilla-Free $\Gamma$** | Hall Row-Col-Row | $\Gamma$ ancilla-free (ours) | $\sim 24L$ | **0** |

**Fair comparison**: 1 FSWAP = 2 CNOT depth for all methods.

In [ ]:
import cirq
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
from typing import Dict, List, Tuple, Sequence
from openfermion.circuits.gates import FSWAP

## 1. Snake Ordering & Grid Utilities

Snake (Jordan-Wigner) ordering: even rows left-to-right, odd rows right-to-left.

In [ ]:
def snake_order_indices(L: int) -> List[int]:
    """Return raster indices in snake order."""
    order = []
    for r in range(L):
        row = [r * L + c for c in range(L)]
        if r % 2 == 1:
            row.reverse()
        order.extend(row)
    return order


def rc_to_snake(r: int, c: int, L: int) -> int:
    if r % 2 == 0:
        return r * L + c
    else:
        return r * L + (L - 1 - c)


def snake_to_rc(idx: int, L: int) -> Tuple[int, int]:
    r = idx // L
    pos_in_row = idx % L
    if r % 2 == 0:
        c = pos_in_row
    else:
        c = L - 1 - pos_in_row
    return r, c


def is_L_row(r: int) -> bool:
    return r % 2 == 0


def sites_between(r1: int, c: int, r2: int, L: int) -> List[int]:
    """Snake-order indices strictly between (r1,c) and (r2,c) for vertical hop."""
    j = rc_to_snake(r1, c, L)
    k = rc_to_snake(r2, c, L)
    lo, hi = min(j, k), max(j, k)
    return list(range(lo + 1, hi))


def make_system_qubits(L: int) -> Dict[Tuple[int, int], cirq.GridQubit]:
    return {(r, c): cirq.GridQubit(r, c) for r in range(L) for c in range(L)}


def make_ancilla_qubits(L: int) -> Dict[int, cirq.NamedQubit]:
    return {r: cirq.NamedQubit(f"anc_r{r}") for r in range(L)}


def validate_permutation(perm, n):
    assert len(perm) == n and set(perm) == set(range(n)), "Invalid permutation"


print("Utilities loaded.")

## 2. Hall's 3-Stage Decomposition (Row-Col-Row)

Decomposes permutation $\pi$ into three stages:
1. **RowA**: within-row permutations (horizontal FSWAP sort)
2. **Col**: within-column permutations (vertical bare FSWAP sort, with $\Gamma$ on each side)
3. **RowB**: within-row permutations (horizontal FSWAP sort)

In the snake JWT, horizontal neighbors are JWT-adjacent, so row FSWAPs need no parity correction.
Only the column stage needs $\Gamma$ to handle the non-local parity string.

In [ ]:
def decompose_permutation_rcr(L: int, perm: Sequence[int]):
    """Decompose permutation into Row-Col-Row stages.

    Returns (s1, s2, s3) where:
      s1[r] = list of length L: within-row permutation for RowA
              s1[r][c] = target column for item at (r,c)
      s2[c] = list of length L: within-column permutation for ColSort
              s2[c][r] = target row for item at (r,c) after RowA
      s3[r] = list of length L: within-row permutation for RowB
              s3[r][c] = target column for item at (r,c) after ColSort
    """
    N = L * L
    validate_permutation(list(perm), N)

    # Build bipartite multigraph: source rows -> destination rows
    G = nx.MultiGraph()
    G.add_nodes_from([f"R{i}" for i in range(L)], bipartite=0)
    G.add_nodes_from([f"D{i}" for i in range(L)], bipartite=1)

    # Each item at (r,c) has destination (dst_r, dst_c)
    edge_data = []  # (R_node, D_node, key, src_r, src_c, dst_r, dst_c)
    for r in range(L):
        for c in range(L):
            idx = r * L + c
            dst = perm[idx]
            dst_r, dst_c = dst // L, dst % L
            key = G.add_edge(f"R{r}", f"D{dst_r}")
            edge_data.append((f"R{r}", f"D{dst_r}", key, r, c, dst_r, dst_c))

    # Decompose L-regular bipartite graph into L perfect matchings
    # Each matching k assigns items to "transit column" k
    matchings = []
    H = G.copy()
    for _ in range(L):
        simp_H = nx.Graph(H.edges())
        matching_dict = nx.bipartite.maximum_matching(
            simp_H, top_nodes=[f"R{i}" for i in range(L)]
        )
        matched_edges = []
        seen = set()
        for u, v in matching_dict.items():
            if (u, v) in seen or (v, u) in seen:
                continue
            seen.add((u, v))
            key = list(H[u][v].keys())[0]
            matched_edges.append((u, v, key))
            H.remove_edge(u, v, key=key)
        matchings.append(matched_edges)

    # Build lookup: (node_u, node_v, key) -> (src_r, src_c, dst_r, dst_c)
    edge_lookup = {}
    for Rn, Dn, key, sr, sc, dr, dc in edge_data:
        edge_lookup[(Rn, Dn, key)] = (sr, sc, dr, dc)
        edge_lookup[(Dn, Rn, key)] = (sr, sc, dr, dc)

    # Matching k: item at (sr, sc) goes to transit column k, destination row dr
    # RowA sends item from column sc to column k within row sr
    s1 = {r: [0] * L for r in range(L)}
    post_rowA = {}  # (row=sr, col=k) -> (dst_r, dst_c)

    for k, matching in enumerate(matchings):
        for u, v, key in matching:
            info_key = (u, v, key) if u.startswith("R") else (v, u, key)
            sr, sc, dr, dc = edge_lookup[info_key]
            s1[sr][sc] = k
            post_rowA[(sr, k)] = (dr, dc)

    # ColSort: after RowA, item at (sr, k) needs to go to row dr
    # s2[c][r] = target row for item at (r, c) after RowA
    s2 = {c: [0] * L for c in range(L)}
    post_colsort = {}  # (row=dr, col=k) -> dst_c

    for (sr, k), (dr, dc) in post_rowA.items():
        s2[k][sr] = dr
        post_colsort[(dr, k)] = dc

    # RowB: after ColSort, item at (dr, k) needs to go to column dc
    s3 = {r: [0] * L for r in range(L)}
    for (dr, k), dc in post_colsort.items():
        s3[dr][k] = dc

    return s1, s2, s3


# Sanity check: verify the decomposition recovers the original permutation
def _verify_decomposition(L, perm, s1, s2, s3):
    for r in range(L):
        for c in range(L):
            # RowA: (r,c) -> (r, s1[r][c])
            c_after_rowA = s1[r][c]
            # ColSort: (r, c_after_rowA) -> (s2[c_after_rowA][r], c_after_rowA)
            r_after_col = s2[c_after_rowA][r]
            # RowB: (r_after_col, c_after_rowA) -> (r_after_col, s3[r_after_col][c_after_rowA])
            c_final = s3[r_after_col][c_after_rowA]
            actual = r_after_col * L + c_final
            expected = perm[r * L + c]
            assert actual == expected, f"Mismatch at ({r},{c}): got {actual}, expected {expected}"


# Test on a few permutations
rng = np.random.default_rng(42)
for L_test in [3, 4, 5]:
    for _ in range(5):
        perm_test = rng.permutation(L_test * L_test).tolist()
        s1, s2, s3 = decompose_permutation_rcr(L_test, perm_test)
        _verify_decomposition(L_test, perm_test, s1, s2, s3)
    print(f"L={L_test}: Hall RCR decomposition verified on 5 random permutations")

## 3. Gamma Operator

$\Gamma$ is a diagonal unitary satisfying the $(\star)$ condition: for every vertical
grid-neighbor pair, $\Gamma \cdot \text{FSWAP}_{\text{bare}} \cdot \Gamma = \text{FSWAP}_{\text{full}}$.

This means one $\Gamma$ before and one after the bare column sort converts all bare FSWAPs
into full fermionic FSWAPs (telescoping property).

Two constructions:
- **Construction 1 (Ancilla)**: Jiang et al., depth $7L - 3$, uses $L$ ancilla qubits
- **Construction 2 (Ancilla-Free)**: Our contribution, depth $9L + 12$, zero ancillas

In [ ]:
def column_parity_cascade_ops(sq: Dict, L: int, inverse: bool = False) -> List[cirq.Operation]:
    """Column parity CNOT cascade.
    Forward: bottom-to-top, CNOT(r+1,c -> r,c) for r from L-2 down to 0.
    Inverse: top-to-bottom, CNOT(r+1,c -> r,c) for r from 0 to L-2."""
    ops = []
    if not inverse:
        for r in range(L - 2, -1, -1):
            for c in range(L):
                ops.append(cirq.CNOT(sq[(r + 1, c)], sq[(r, c)]))
    else:
        for r in range(L - 1):
            for c in range(L):
                ops.append(cirq.CNOT(sq[(r + 1, c)], sq[(r, c)]))
    return ops

In [ ]:
# Construction 1: Gamma with Ancillas (Jiang et al.)
# CNOT depth: 7L - 3, Gate count: O(N), Ancillas: L

def build_stage_B_ops(sq: Dict, aq: Dict, L: int) -> List[cirq.Operation]:
    """Stage B: parity-basis CZ sweep (R->L). Depth 3L."""
    ops = []
    for p in range(L - 1, -1, -1):
        for r in range(0, L, 2):
            if r + 2 <= L - 1:
                ops.append(cirq.CZ(sq[(r, p)], aq[r + 2]))
            if r >= 2:
                ops.append(cirq.CZ(sq[(r, p)], aq[r]))
        for r in range(L):
            ops.append(cirq.CNOT(sq[(r, p)], aq[r]))
    return ops


def ancilla_column_cascade_ops(aq: Dict, L: int, inverse: bool = False) -> List[cirq.Operation]:
    """CNOT cascade on ancilla column. Depth L-1."""
    if not inverse:
        return [cirq.CNOT(aq[r + 1], aq[r]) for r in range(L - 2, -1, -1)]
    else:
        return [cirq.CNOT(aq[r + 1], aq[r]) for r in range(L - 1)]


def build_stage_D_ops(sq: Dict, aq: Dict, L: int) -> List[cirq.Operation]:
    """Stage D: original-basis CZ sweep (L->R). Depth 2L+1."""
    ops = []
    for p in range(L):
        for r in range(L):
            ops.append(cirq.CNOT(sq[(r, p)], aq[r]))
        for r in range(0, L, 2):
            if r + 1 < L:
                ops.append(cirq.CZ(sq[(r, p)], aq[r]))
                ops.append(cirq.CZ(sq[(r, p)], aq[r + 1]))
            else:
                ops.append(cirq.CZ(sq[(r, p)], aq[r]))
    return ops


def build_gamma_with_ancillas(L: int):
    """Build Gamma with ancillas. Returns (circuit, sys_list, anc_list).
    CNOT depth: 7L - 3."""
    sq = make_system_qubits(L)
    aq = make_ancilla_qubits(L)
    ops = []
    ops.extend(column_parity_cascade_ops(sq, L, inverse=False))   # Stage A
    ops.extend(build_stage_B_ops(sq, aq, L))                      # Stage B
    ops.extend(column_parity_cascade_ops(sq, L, inverse=True))    # Stage C (sys)
    ops.extend(ancilla_column_cascade_ops(aq, L, inverse=True))   # Stage C (anc)
    ops.extend(build_stage_D_ops(sq, aq, L))                      # Stage D
    circuit = cirq.Circuit(ops)
    sys_list = [sq[(r, c)] for r in range(L) for c in range(L)]
    anc_list = [aq[r] for r in range(L)]
    return circuit, sys_list, anc_list


# Depth check
for L_test in [3, 5, 7]:
    c1, _, _ = build_gamma_with_ancillas(L_test)
    expected = 7 * L_test - 3
    assert len(c1) == expected, f"L={L_test}: got {len(c1)}, expected {expected}"
    print(f"Gamma (ancilla) L={L_test}: depth = {len(c1)} = 7L-3 = {expected} ✓")

In [ ]:
# Construction 2: Ancilla-Free Gamma
# CNOT depth: 9L + 12 (exact for L >= 5), Gate count: O(N), Ancillas: 0

def suffix_cascade_ops(sq: Dict, r: int, L: int) -> List[cirq.Operation]:
    return [cirq.CNOT(sq[(r, c + 1)], sq[(r, c)]) for c in range(L - 2, -1, -1)]

def undo_suffix_cascade_ops(sq: Dict, r: int, L: int) -> List[cirq.Operation]:
    return [cirq.CNOT(sq[(r, c + 1)], sq[(r, c)]) for c in range(L - 1)]

def prefix_cascade_ops(sq: Dict, r: int, L: int) -> List[cirq.Operation]:
    return [cirq.CNOT(sq[(r, c - 1)], sq[(r, c)]) for c in range(1, L)]

def undo_prefix_cascade_ops(sq: Dict, r: int, L: int) -> List[cirq.Operation]:
    return [cirq.CNOT(sq[(r, c - 1)], sq[(r, c)]) for c in range(L - 1, 0, -1)]


def same_row_T_ops(sq: Dict, r: int, L: int) -> List[cirq.Operation]:
    """Primitive 1: T(x,x) for row r. Depth 2L."""
    ops = []
    ops.extend(suffix_cascade_ops(sq, r, L))
    for c in range(L - 1):
        ops.append(cirq.CZ(sq[(r, c)], sq[(r, c + 1)]))
    ops.extend(undo_suffix_cascade_ops(sq, r, L))
    for c in range(1, L, 2):
        ops.append(cirq.Z(sq[(r, c)]))
    return ops


def cross_row_adjacent_T_ops(sq: Dict, r1: int, r2: int, L: int) -> List[cirq.Operation]:
    """Primitive 2: T(x,y) for adjacent rows r1, r2. Depth 2L."""
    ops = []
    ops.extend(prefix_cascade_ops(sq, r1, L))
    for c in range(L):
        ops.append(cirq.CZ(sq[(r1, c)], sq[(r2, c)]))
    ops.extend(undo_prefix_cascade_ops(sq, r1, L))
    for c in range(L):
        ops.append(cirq.CZ(sq[(r1, c)], sq[(r2, c)]))
    return ops


def skip_row_T_ops(sq: Dict, r1: int, r2: int, r_mid: int, L: int) -> List[cirq.Operation]:
    """Primitive 3: T(x,y) for rows 2 apart, routing through r_mid. Depth 2L+4."""
    ops = []
    ops.extend(prefix_cascade_ops(sq, r1, L))
    for c in range(L):
        ops.append(cirq.CZ(sq[(r_mid, c)], sq[(r2, c)]))
        ops.append(cirq.CNOT(sq[(r1, c)], sq[(r_mid, c)]))
        ops.append(cirq.CZ(sq[(r_mid, c)], sq[(r2, c)]))
        ops.append(cirq.CNOT(sq[(r1, c)], sq[(r_mid, c)]))
    ops.extend(undo_prefix_cascade_ops(sq, r1, L))
    for c in range(L):
        ops.append(cirq.CZ(sq[(r_mid, c)], sq[(r2, c)]))
        ops.append(cirq.CNOT(sq[(r1, c)], sq[(r_mid, c)]))
        ops.append(cirq.CZ(sq[(r_mid, c)], sq[(r2, c)]))
        ops.append(cirq.CNOT(sq[(r1, c)], sq[(r_mid, c)]))
    return ops


def build_gamma_ancilla_free(L: int):
    """Build ancilla-free Gamma. Returns (circuit, sys_list).
    CNOT depth: 9L + 12 (exact for L >= 5)."""
    sq = make_system_qubits(L)
    ops = []

    # Phase 1: column parity cascade forward
    ops.extend(column_parity_cascade_ops(sq, L, inverse=False))

    # Phase 2a: D_B same-row terms for even rows r >= 2
    for r in range(2, L, 2):
        ops.extend(same_row_T_ops(sq, r, L))

    # Phase 2b: D_B skip-row terms (2-round scheduling)
    skip_rows = [r for r in range(0, L, 2) if r + 2 <= L - 1]
    for r in skip_rows[0::2]:   # Round 1
        ops.extend(skip_row_T_ops(sq, r, r + 2, r + 1, L))
    for r in skip_rows[1::2]:   # Round 2
        ops.extend(skip_row_T_ops(sq, r, r + 2, r + 1, L))

    # Phase 3: column parity cascade inverse
    ops.extend(column_parity_cascade_ops(sq, L, inverse=True))

    # Phase 4a: D_D same-row terms for all even rows
    for r in range(0, L, 2):
        ops.extend(same_row_T_ops(sq, r, L))

    # Phase 4b: D_D cross-row terms for right-closed pairs
    for r in range(0, L - 1, 2):
        ops.extend(cross_row_adjacent_T_ops(sq, r, r + 1, L))

    circuit = cirq.Circuit(ops)
    sys_list = [sq[(r, c)] for r in range(L) for c in range(L)]
    return circuit, sys_list


# Depth check
for L_test in [5, 7, 9]:
    c2, _ = build_gamma_ancilla_free(L_test)
    expected = 9 * L_test + 12
    assert len(c2) == expected, f"L={L_test}: got {len(c2)}, expected {expected}"
    print(f"Gamma (ancilla-free) L={L_test}: depth = {len(c2)} = 9L+12 = {expected} ✓")

## 4. FSWAP Odd-Even Transposition Sort

Standard odd-even transposition sort using FSWAP gates:
- Each FSWAP = SWAP + CZ = 2 CNOT depth (iSWAP equivalent)
- At most $L$ rounds sort any permutation on $L$ elements
- Used for: row stages (Methods 1/2), column stage (bare FSWAP between $\Gamma$'s), and entire baseline

In [ ]:
def fswap_odd_even_sort_ops(qubits: list, perm: list) -> List[cirq.Operation]:
    """Build FSWAP-based odd-even transposition sort.

    Args:
        qubits: list of cirq qubits in order
        perm: permutation as list where perm[i] = destination of item at position i

    Returns:
        list of FSWAP operations implementing the sort
    """
    n = len(qubits)
    if n <= 1:
        return []

    ops = []
    current = list(perm)  # current[i] = which destination the item at position i wants

    for round_num in range(n):
        parity = round_num % 2  # 0 = even pairs (0,1),(2,3),...  1 = odd pairs (1,2),(3,4),...
        swapped = False
        for i in range(parity, n - 1, 2):
            # Compare: item at position i wants current[i], item at i+1 wants current[i+1]
            # Swap if item at i needs to go right of item at i+1
            if current[i] > current[i + 1]:
                ops.append(FSWAP.on(qubits[i], qubits[i + 1]))
                current[i], current[i + 1] = current[i + 1], current[i]
                swapped = True
        if not swapped and round_num > 0:
            break  # Already sorted

    return ops


# Test: verify FSWAP sort produces correct permutation
def _test_fswap_sort():
    rng = np.random.default_rng(42)
    for n in [3, 4, 5, 6]:
        for _ in range(10):
            perm = rng.permutation(n).tolist()
            qubits = cirq.LineQubit.range(n)
            ops = fswap_odd_even_sort_ops(list(qubits), perm)
            # The sort should transform perm to identity
            check = list(perm)
            for op in ops:
                q0, q1 = op.qubits
                i, j = qubits.index(q0), qubits.index(q1)
                assert abs(i - j) == 1
                check[i], check[j] = check[j], check[i]
            assert check == list(range(n)), f"Sort failed for perm={perm}, got {check}"
    print("FSWAP odd-even sort verified on random permutations")

_test_fswap_sort()

## 5. Full Circuit Builders

Each method builds the complete fermionic permutation circuit:

**Methods 1 & 2**: RowA (FSWAP sort) $\to$ $\Gamma$ $\to$ bare column FSWAP sort $\to$ $\Gamma$ $\to$ RowB (FSWAP sort)

**Baseline**: 1D snake FSWAP sort over all $L^2$ qubits

In [ ]:
def build_method1_circuit(L: int, perm: list):
    """Ancilla-Gamma method: Hall RCR + ancilla-based Gamma.
    Returns (circuit, sys_qubits, anc_qubits)."""
    N = L * L
    validate_permutation(perm, N)
    s1, s2, s3 = decompose_permutation_rcr(L, perm)

    sq = make_system_qubits(L)
    aq = make_ancilla_qubits(L)

    # Pre-build Gamma ops (reused for both applications since Gamma = Gamma dagger)
    gamma_ops = []
    gamma_ops.extend(column_parity_cascade_ops(sq, L, inverse=False))
    gamma_ops.extend(build_stage_B_ops(sq, aq, L))
    gamma_ops.extend(column_parity_cascade_ops(sq, L, inverse=True))
    gamma_ops.extend(ancilla_column_cascade_ops(aq, L, inverse=True))
    gamma_ops.extend(build_stage_D_ops(sq, aq, L))

    # Build each stage as a separate circuit to preserve stage boundaries
    rowA_ops = []
    for r in range(L):
        row_qs = [sq[(r, c)] for c in range(L)]
        rowA_ops.extend(fswap_odd_even_sort_ops(row_qs, s1[r]))

    col_ops = []
    for c in range(L):
        col_qs = [sq[(r, c)] for r in range(L)]
        col_ops.extend(fswap_odd_even_sort_ops(col_qs, s2[c]))

    rowB_ops = []
    for r in range(L):
        row_qs = [sq[(r, c)] for c in range(L)]
        rowB_ops.extend(fswap_odd_even_sort_ops(row_qs, s3[r]))

    circuit = (cirq.Circuit(rowA_ops)
             + cirq.Circuit(gamma_ops)
             + cirq.Circuit(col_ops)
             + cirq.Circuit(gamma_ops)
             + cirq.Circuit(rowB_ops))

    sys_list = [sq[(r, c)] for r in range(L) for c in range(L)]
    anc_list = [aq[r] for r in range(L)]
    return circuit, sys_list, anc_list


print("Method 1 (Ancilla Gamma) builder defined.")

In [ ]:
def build_method2_circuit(L: int, perm: list):
    """Ancilla-Free Gamma method: Hall RCR + ancilla-free Gamma.
    Returns (circuit, sys_qubits)."""
    N = L * L
    validate_permutation(perm, N)
    s1, s2, s3 = decompose_permutation_rcr(L, perm)

    sq = make_system_qubits(L)

    gamma_circ, _ = build_gamma_ancilla_free(L)

    # Build each stage as a separate circuit to preserve stage boundaries
    rowA_ops = []
    for r in range(L):
        row_qs = [sq[(r, c)] for c in range(L)]
        rowA_ops.extend(fswap_odd_even_sort_ops(row_qs, s1[r]))

    col_ops = []
    for c in range(L):
        col_qs = [sq[(r, c)] for r in range(L)]
        col_ops.extend(fswap_odd_even_sort_ops(col_qs, s2[c]))

    rowB_ops = []
    for r in range(L):
        row_qs = [sq[(r, c)] for c in range(L)]
        rowB_ops.extend(fswap_odd_even_sort_ops(row_qs, s3[r]))

    circuit = (cirq.Circuit(rowA_ops)
             + gamma_circ
             + cirq.Circuit(col_ops)
             + gamma_circ
             + cirq.Circuit(rowB_ops))

    sys_list = [sq[(r, c)] for r in range(L) for c in range(L)]
    return circuit, sys_list


print("Method 2 (Ancilla-Free Gamma) builder defined.")

In [ ]:
def build_baseline_circuit(L: int, perm_raster: list):
    """Baseline: 1D snake ordering + full FSWAP odd-even sort.
    Returns (circuit, qubit_list_snake_order)."""
    N = L * L
    validate_permutation(perm_raster, N)

    # Convert raster permutation to snake-order permutation
    snake = snake_order_indices(L)
    r2s = {r_idx: s_pos for s_pos, r_idx in enumerate(snake)}

    perm_snake = [0] * N
    for src_r in range(N):
        dst_r = perm_raster[src_r]
        perm_snake[r2s[src_r]] = r2s[dst_r]

    # Use GridQubits in snake order for the baseline
    qubits = [cirq.GridQubit(*snake_to_rc(i, L)) for i in range(N)]
    ops = fswap_odd_even_sort_ops(qubits, perm_snake)

    circuit = cirq.Circuit(ops)
    return circuit, qubits


print("Baseline (1D Snake) builder defined.")

## 6. Verification

### Methodology

**Tier 1 -- Gamma operator correctness** (classical Clifford simulation):
Since $\Gamma$ uses only CNOT, CZ, and Z gates (all Clifford), we simulate it classically:
track computational-basis bits through CNOTs and accumulate phase from CZ/Z gates.
This is $O(\text{gates})$ per basis state and scales to any qubit count.

We verify:
1. **Construction agreement**: Both Gamma constructions produce identical diagonal phases
   on 500 random basis states for $L = 3, \ldots, 9$.
2. **Property ($\star$)**: For every vertical grid-neighbor pair and every sampled basis state,
   $\gamma_s \cdot \gamma_{s'} = (-1)^P$ where $P$ is the parity of JWT-intermediate qubits.
   Tested on 300 random states per $L$.

**Tier 2 -- End-to-end circuit equivalence** (state-vector simulation):
For small $L$ (3 and 4), we build circuits for all three methods, simulate with `cirq.Simulator`,
and compare output density matrices. Tests use:
- **Random permutations**: 10 trials per $L$
- **Structured permutations**: identity, transpose $(r,c) \to (c,r)$, and reverse

**Depth counting**:
Circuit depth = number of moments after Cirq's greedy scheduler.
Each CZ, CNOT, or FSWAP occupies 1 moment as a 2-qubit gate.
For CNOT-equivalent depth: FSWAP $\to$ 2 CNOTs, CZ $\to$ 1 CNOT (via H-CNOT-H).

In [ ]:
# --- Structured permutations ---

def structured_permutations(L: int) -> Dict[str, list]:
    """Return named structured permutations for an L x L grid."""
    N = L * L
    return {
        'identity': list(range(N)),
        'transpose': [(c * L + r) for r in range(L) for c in range(L)],
        'reverse': list(range(N - 1, -1, -1)),
    }


# --- Classical Clifford simulation for Gamma verification ---

def classical_sim_phase(ops_list, qubit_to_idx, n_qubits, basis_state_bits):
    """Simulate Clifford circuit on a computational basis state.
    Returns (phase, final_bits) where phase is +1 or -1."""
    bits = list(basis_state_bits)
    phase = 0
    for op in ops_list:
        gate = op.gate
        qubits = op.qubits
        if isinstance(gate, cirq.ops.common_gates.CNotPowGate) and gate.exponent == 1:
            ctrl_idx = qubit_to_idx[qubits[0]]
            tgt_idx = qubit_to_idx[qubits[1]]
            bits[tgt_idx] ^= bits[ctrl_idx]
        elif isinstance(gate, cirq.ops.common_gates.CZPowGate) and gate.exponent == 1:
            a_idx = qubit_to_idx[qubits[0]]
            b_idx = qubit_to_idx[qubits[1]]
            phase ^= (bits[a_idx] & bits[b_idx])
        elif isinstance(gate, cirq.ops.common_gates.ZPowGate) and gate.exponent == 1:
            idx = qubit_to_idx[qubits[0]]
            phase ^= bits[idx]
        else:
            raise ValueError(f"Unsupported gate in Gamma: {gate}")
    return (-1)**phase, bits


def get_phase_classical(circuit, qubit_order, basis_state):
    n = len(qubit_order)
    q2i = {q: i for i, q in enumerate(qubit_order)}
    bits = [(basis_state >> (n - 1 - i)) & 1 for i in range(n)]
    all_ops = [op for moment in circuit for op in moment]
    phase, _ = classical_sim_phase(all_ops, q2i, n, bits)
    return phase


def get_phase_classical_with_ancillas(circuit, sys_qubits, anc_qubits, basis_state):
    n_sys = len(sys_qubits)
    n_anc = len(anc_qubits)
    all_qubits = sys_qubits + anc_qubits
    q2i = {q: i for i, q in enumerate(all_qubits)}
    bits = [(basis_state >> (n_sys - 1 - i)) & 1 for i in range(n_sys)] + [0] * n_anc
    all_ops = [op for moment in circuit for op in moment]
    phase, final_bits = classical_sim_phase(all_ops, q2i, n_sys + n_anc, bits)
    for i in range(n_anc):
        assert final_bits[n_sys + i] == 0, f"Ancilla {i} not disentangled!"
    return phase


def verify_property_star(phase_fn, L, num_samples=300, seed=42):
    """Verify property (star) on sampled basis states."""
    rng = np.random.default_rng(seed)
    N = L * L
    checked, passed = 0, 0
    for _ in range(num_samples):
        bits = rng.integers(0, 2, size=N)
        s_idx = sum(int(b) << (N - 1 - i) for i, b in enumerate(bits))
        gamma_s = phase_fn(s_idx)
        for r in range(L - 1):
            for c in range(L):
                if bits[r * L + c] == bits[(r + 1) * L + c]:
                    continue
                s_prime_idx = s_idx ^ (1 << (N - 1 - r * L - c)) ^ (1 << (N - 1 - (r + 1) * L - c))
                gamma_s_prime = phase_fn(s_prime_idx)
                between = sites_between(r, c, r + 1, L)
                P = 0
                for site_snake in between:
                    sr, sc = snake_to_rc(site_snake, L)
                    P ^= int(bits[sr * L + sc])
                expected = (-1) ** P
                actual = gamma_s * gamma_s_prime
                checked += 1
                if abs(actual - expected) < 1e-6:
                    passed += 1
    return checked, passed


print("Verification utilities defined.")

In [ ]:
print("=" * 70)
print("  GAMMA OPERATOR VERIFICATION")
print("  Testing both constructions match + property (star) holds")
print("=" * 70)

all_passed = True
for L_test in [3, 4, 5, 6, 7, 8, 9]:
    c1, sys1, anc1 = build_gamma_with_ancillas(L_test)
    c2, sys2 = build_gamma_ancilla_free(L_test)
    N = L_test * L_test

    rng = np.random.default_rng(42 + L_test)
    mismatches = 0
    for _ in range(500):
        bits = rng.integers(0, 2, size=N)
        s = sum(int(b) << (N - 1 - i) for i, b in enumerate(bits))
        p1 = get_phase_classical_with_ancillas(c1, sys1, anc1, s)
        p2 = get_phase_classical(c2, sys2, s)
        if abs(p1 - p2) > 1e-6:
            mismatches += 1

    ch, pa = verify_property_star(
        lambda s, c=c2, sq=sys2: get_phase_classical(c, sq, s),
        L_test, num_samples=300, seed=42 + L_test
    )
    star_ok = (pa == ch)
    match_ok = (mismatches == 0)
    if not match_ok or not star_ok:
        all_passed = False

    status = "PASS" if (match_ok and star_ok) else "FAIL"
    print(f"  L={L_test:2d} (N={N:3d}): match {500 - mismatches}/500 | "
          f"Property (star): {pa}/{ch} | {status}")

print("-" * 70)
print(f"  Overall: {'ALL PASSED' if all_passed else 'SOME FAILED'}")
print("=" * 70)

In [ ]:
def verify_end_to_end(L, num_random_trials=8, seed=42):
    """Verify all 3 methods agree for given L."""
    rng = np.random.default_rng(seed)
    N = L * L
    sim = cirq.Simulator()
    canonical_qubits = [cirq.GridQubit(r, c) for r in range(L) for c in range(L)]
    results = []

    perms = []
    sp = structured_permutations(L)
    for name, p in sp.items():
        perms.append((name, p))
    for t in range(num_random_trials):
        perms.append((f"random_{t}", rng.permutation(N).tolist()))

    for perm_name, perm in perms:
        circ_m1, sys_m1, anc_m1 = build_method1_circuit(L, perm)
        circ_m2, sys_m2 = build_method2_circuit(L, perm)
        circ_base, qubits_base = build_baseline_circuit(L, perm)

        init_bits = rng.integers(0, 2, size=N).tolist()

        # Method 2
        qubit_order_m2 = canonical_qubits
        init_state_m2 = np.zeros(2**N, dtype=complex)
        idx_m2 = sum(init_bits[i] << (N - 1 - i) for i in range(N))
        init_state_m2[idx_m2] = 1.0
        circ_m2_full = circ_m2 + cirq.Circuit([cirq.I(q) for q in qubit_order_m2])
        result_m2 = sim.simulate(circ_m2_full, qubit_order=qubit_order_m2,
                                  initial_state=init_state_m2)
        state_m2 = result_m2.final_state_vector

        # Method 1
        anc_qubits_sorted = sorted(anc_m1, key=str)
        all_qubits_m1 = canonical_qubits + anc_qubits_sorted
        n_total_m1 = len(all_qubits_m1)
        bits_m1 = init_bits + [0] * len(anc_m1)
        idx_m1 = sum(bits_m1[i] << (n_total_m1 - 1 - i) for i in range(n_total_m1))
        init_state_m1 = np.zeros(2**n_total_m1, dtype=complex)
        init_state_m1[idx_m1] = 1.0
        circ_m1_full = circ_m1 + cirq.Circuit([cirq.I(q) for q in all_qubits_m1])
        result_m1 = sim.simulate(circ_m1_full, qubit_order=all_qubits_m1,
                                  initial_state=init_state_m1)
        state_m1_full = result_m1.final_state_vector

        n_anc = len(anc_m1)
        sys_positions = list(range(N))
        dm_m1_full = np.outer(state_m1_full, np.conj(state_m1_full))
        dm_m1 = cirq.partial_trace(dm_m1_full.reshape([2] * n_total_m1 * 2),
                                    keep_indices=sys_positions)
        dm_m1_flat = dm_m1.reshape(2**N, 2**N)
        dm_m2 = np.outer(state_m2, np.conj(state_m2))

        # Baseline
        init_state_base = np.zeros(2**N, dtype=complex)
        init_state_base[idx_m2] = 1.0
        circ_base_full = circ_base + cirq.Circuit([cirq.I(q) for q in canonical_qubits])
        result_base = sim.simulate(circ_base_full, qubit_order=canonical_qubits,
                                    initial_state=init_state_base)
        state_base = result_base.final_state_vector
        dm_base = np.outer(state_base, np.conj(state_base))

        fid_12 = np.real(np.trace(dm_m1_flat @ dm_m2))
        fid_1b = np.real(np.trace(dm_m1_flat @ dm_base))
        fid_2b = np.real(np.trace(dm_m2 @ dm_base))
        ok = abs(fid_12 - 1) < 1e-6 and abs(fid_1b - 1) < 1e-6 and abs(fid_2b - 1) < 1e-6
        results.append((perm_name, fid_12, fid_1b, fid_2b, ok))

    return results


print("=" * 70)
print("  END-TO-END VERIFICATION")
print("  Comparing all 3 methods on random + structured permutations")
print("=" * 70)

for L_test in [3]:
    print(f"\n--- L = {L_test} (N = {L_test**2}) ---")
    results = verify_end_to_end(L_test, num_random_trials=8, seed=42 + L_test)
    n_pass = sum(1 for r in results if r[4])
    n_total = len(results)

    for name, f12, f1b, f2b, ok in results:
        tag = "PASS" if ok else "FAIL"
        if name.startswith("random") and ok:
            continue
        print(f"  {name:12s}: fid(M1,M2)={f12:.6f}  fid(M1,Base)={f1b:.6f}  "
              f"fid(M2,Base)={f2b:.6f}  [{tag}]")

    random_results = [r for r in results if r[0].startswith("random")]
    random_pass = sum(1 for r in random_results if r[4])
    print(f"  random:      {random_pass}/{len(random_results)} passed")
    print(f"  TOTAL:       {n_pass}/{n_total} passed")

print("\n" + "=" * 70)

In [ ]:
print("=" * 70)
print("  GAMMA OPERATOR DEPTH ANALYSIS")
print("=" * 70)
print()
print(f"{'L':>3} | {'Ancilla depth':>14} | {'7L-3':>6} | {'match':>5} | "
      f"{'AncFree depth':>14} | {'9L+12':>6} | {'match':>5}")
print("-" * 80)

for L_test in [3, 5, 7, 9, 11, 15, 20]:
    c1, _, _ = build_gamma_with_ancillas(L_test)
    c2, _ = build_gamma_ancilla_free(L_test)
    d1 = len(c1)
    d2 = len(c2)
    e1 = 7 * L_test - 3
    e2 = 9 * L_test + 12
    m1 = "yes" if d1 == e1 else "NO"
    m2 = "yes" if d2 == e2 else "NO" if L_test >= 5 else "n/a"
    print(f"{L_test:3d} | {d1:14d} | {e1:6d} | {m1:>5} | {d2:14d} | {e2:6d} | {m2:>5}")

print()
print("Note: 9L+12 exact for L >= 5. 7L-3 exact for all L >= 3.")

In [ ]:
print("=" * 70)
print("  PER-STAGE DEPTH BREAKDOWN (transpose permutation)")
print("=" * 70)

def measure_stage_depth(ops_list):
    if not ops_list:
        return 0
    return len(cirq.Circuit(ops_list))

header_printed = False
for L_test in [3, 5, 7, 9, 11, 15]:
    perm = structured_permutations(L_test)['transpose']
    s1, s2, s3 = decompose_permutation_rcr(L_test, perm)
    sq = make_system_qubits(L_test)

    rowA_ops = []
    for r in range(L_test):
        rowA_ops.extend(fswap_odd_even_sort_ops([sq[(r,c)] for c in range(L_test)], s1[r]))
    col_ops = []
    for c in range(L_test):
        col_ops.extend(fswap_odd_even_sort_ops([sq[(r,c)] for r in range(L_test)], s2[c]))
    rowB_ops = []
    for r in range(L_test):
        rowB_ops.extend(fswap_odd_even_sort_ops([sq[(r,c)] for c in range(L_test)], s3[r]))

    gamma1_circ, _, _ = build_gamma_with_ancillas(L_test)
    gamma2_circ, _ = build_gamma_ancilla_free(L_test)

    d_rA = measure_stage_depth(rowA_ops)
    d_col = measure_stage_depth(col_ops)
    d_rB = measure_stage_depth(rowB_ops)
    d_g1 = len(gamma1_circ)
    d_g2 = len(gamma2_circ)

    circ_m1, _, _ = build_method1_circuit(L_test, perm)
    circ_m2, _ = build_method2_circuit(L_test, perm)
    circ_base, _ = build_baseline_circuit(L_test, perm)

    sum_m1 = d_rA + d_g1 + d_col + d_g1 + d_rB
    sum_m2 = d_rA + d_g2 + d_col + d_g2 + d_rB

    if not header_printed:
        print(f"{'L':>3} | {'Method':>10} | {'RowA':>4} | {'G':>4} | {'Col':>4} | "
              f"{'G':>4} | {'RowB':>4} | {'Sum':>4} | {'Tot':>4} | {'Ovlp':>4} | {'Base':>4}")
        print("-" * 80)
        header_printed = True

    print(f"{L_test:3d} | {'Ancilla':>10} | {d_rA:4d} | {d_g1:4d} | {d_col:4d} | "
          f"{d_g1:4d} | {d_rB:4d} | {sum_m1:4d} | {len(circ_m1):4d} | "
          f"{sum_m1 - len(circ_m1):4d} | {len(circ_base):4d}")
    print(f"    | {'Anc-Free':>10} | {d_rA:4d} | {d_g2:4d} | {d_col:4d} | "
          f"{d_g2:4d} | {d_rB:4d} | {sum_m2:4d} | {len(circ_m2):4d} | "
          f"{sum_m2 - len(circ_m2):4d} |")

print()
print("Ovlp = Sum - Tot (inter-stage scheduling overlap)")
print("Gamma depth is permutation-independent; sort depths depend on permutation.")

## 7. Resource Analysis

**Depth metric**: Number of circuit moments (Cirq greedy scheduler).
Each FSWAP, CZ, or CNOT = one 2-qubit gate occupying one moment.

**CNOT-equivalent depth**: After lowering FSWAP $\to$ 2 CNOTs, CZ $\to$ 1 CNOT (H+CNOT+H).
Since CZ decomposes to exactly 1 CNOT, the CNOT depth equals gate depth for
CZ/CNOT-only circuits (like $\Gamma$). For FSWAP-containing stages (sorting),
CNOT depth $\approx$ 2$\times$ gate depth.

**Multi-permutation sampling**: Each $L$ value is tested with 3 random permutations +
transpose + reverse = 5 total. Mean $\pm$ std reported.

In [ ]:
def _is_fswap(gate):
    from openfermion.circuits.gates import FSwapPowGate
    return isinstance(gate, FSwapPowGate)


def count_resources(circuit, method_name, L, n_ancillas=0):
    """Count gate depth and CNOT-equivalent depth.

    Gate depth: moments containing any 2-qubit gate.
    CNOT depth: FSWAP moments count as 2 (since FSWAP = 2 CNOTs on same pair),
                CZ/CNOT moments count as 1 (CZ = H+CNOT+H = 1 CNOT depth).
    This avoids the flattening artifact where re-scheduling lowered ops
    loses inter-round parallelism."""
    gate_depth = 0
    cnot_depth = 0
    total_2q = 0

    for moment in circuit:
        has_fswap = False
        has_other_2q = False
        for op in moment:
            if len(op.qubits) >= 2:
                total_2q += 1
                if _is_fswap(op.gate):
                    has_fswap = True
                else:
                    has_other_2q = True

        if has_fswap or has_other_2q:
            gate_depth += 1
        if has_fswap:
            cnot_depth += 2   # FSWAP = iSWAP ~ 2 CNOTs (sequential on same pair)
        elif has_other_2q:
            cnot_depth += 1   # CZ or CNOT = 1 CNOT depth

    # Total CNOTs: each FSWAP -> 2 CNOTs, each CZ -> 1 CNOT, each CNOT -> 1 CNOT
    total_cnots = 0
    for moment in circuit:
        for op in moment:
            if len(op.qubits) >= 2:
                if _is_fswap(op.gate):
                    total_cnots += 2
                else:
                    total_cnots += 1

    return {
        'method': method_name, 'L': L, 'N': L*L, 'ancillas': n_ancillas,
        'gate_depth': gate_depth, 'cnot_depth': cnot_depth,
        'total_2q_gates': total_2q, 'total_cnots': total_cnots,
    }


print("Resource counting utilities defined.")

In [ ]:
def resource_comparison_multi(L_values, n_random=3, seed=42):
    all_rows = []
    for L in L_values:
        rng = np.random.default_rng(seed + L)
        sp = structured_permutations(L)
        perms = [(name, p) for name, p in sp.items() if name != 'identity']
        for t in range(n_random):
            perms.append((f"random_{t}", rng.permutation(L * L).tolist()))

        for perm_name, perm in perms:
            circ_base, _ = build_baseline_circuit(L, perm)
            circ_m1, _, anc_m1 = build_method1_circuit(L, perm)
            circ_m2, _ = build_method2_circuit(L, perm)

            r = count_resources(circ_base, 'Baseline', L, 0); r['perm'] = perm_name; all_rows.append(r)
            r = count_resources(circ_m1, 'Ancilla Gamma', L, len(anc_m1)); r['perm'] = perm_name; all_rows.append(r)
            r = count_resources(circ_m2, 'AncFree Gamma', L, 0); r['perm'] = perm_name; all_rows.append(r)

        print(f"  L={L} done ({len(perms)} permutations)")
    return pd.DataFrame(all_rows)


L_values = [3, 4, 5, 7, 9, 11, 15]
print("=" * 70)
print("  RESOURCE COMPARISON (multi-permutation)")
print(f"  L values: {L_values}")
print(f"  Per L: 3 random + transpose + reverse = 5 permutations")
print("=" * 70)

df_all = resource_comparison_multi(L_values, n_random=3)

print()
print(f"{'L':>3} | {'N':>4} | {'Method':>14} | {'GateDepth':>14} | {'CNOTDepth':>14} | "
      f"{'2Q Gates':>14} | {'Anc':>3}")
print("-" * 85)

for L in L_values:
    for method in ['Baseline', 'Ancilla Gamma', 'AncFree Gamma']:
        sub = df_all[(df_all['L'] == L) & (df_all['method'] == method)]
        gd_m, gd_s = sub['gate_depth'].mean(), sub['gate_depth'].std()
        cd_m, cd_s = sub['cnot_depth'].mean(), sub['cnot_depth'].std()
        tq_m, tq_s = sub['total_2q_gates'].mean(), sub['total_2q_gates'].std()
        anc = sub['ancillas'].iloc[0]
        print(f"{L:3d} | {L*L:4d} | {method:>14} | {gd_m:6.1f}+/-{gd_s:4.1f} | "
              f"{cd_m:6.1f}+/-{cd_s:4.1f} | {tq_m:7.1f}+/-{tq_s:5.1f} | {anc:3d}")
    print("-" * 85)

In [ ]:
agg = df_all.groupby(['L', 'method']).agg(
    gate_depth_mean=('gate_depth', 'mean'), gate_depth_std=('gate_depth', 'std'),
    cnot_depth_mean=('cnot_depth', 'mean'), cnot_depth_std=('cnot_depth', 'std'),
    total_2q_mean=('total_2q_gates', 'mean'), total_2q_std=('total_2q_gates', 'std'),
).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {'Baseline': 'gray', 'Ancilla Gamma': 'blue', 'AncFree Gamma': 'green'}
markers = {'Baseline': 's', 'Ancilla Gamma': 'o', 'AncFree Gamma': '^'}
labels = {'Baseline': 'Baseline (1D Snake)', 'Ancilla Gamma': 'Ancilla $\\Gamma$',
          'AncFree Gamma': 'Ancilla-Free $\\Gamma$'}

for method in ['Baseline', 'Ancilla Gamma', 'AncFree Gamma']:
    sub = agg[agg['method'] == method]
    Ls = sub['L'].values
    axes[0,0].errorbar(Ls, sub['gate_depth_mean'], yerr=sub['gate_depth_std'],
                        fmt=f'{markers[method]}-', color=colors[method],
                        label=labels[method], markersize=5, capsize=3)
    axes[0,1].errorbar(Ls, sub['cnot_depth_mean'], yerr=sub['cnot_depth_std'],
                        fmt=f'{markers[method]}-', color=colors[method],
                        label=labels[method], markersize=5, capsize=3)
    axes[1,0].errorbar(Ls, sub['total_2q_mean'], yerr=sub['total_2q_std'],
                        fmt=f'{markers[method]}-', color=colors[method],
                        label=labels[method], markersize=5, capsize=3)
    axes[1,1].plot(Ls, sub['gate_depth_mean'] / Ls, f'{markers[method]}-',
                    color=colors[method], label=labels[method], markersize=5)

L_th = np.linspace(3, max(L_values), 100)
axes[0,0].plot(L_th, L_th**2, '--', color='gray', alpha=0.3, label='$L^2$')
axes[0,1].plot(L_th, 2*L_th**2, '--', color='gray', alpha=0.3, label='$2L^2$')

for ax in axes.flat:
    ax.set_xlabel('L (grid side)'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
axes[0,0].set_ylabel('Gate Depth'); axes[0,0].set_title('Gate Depth vs Grid Size')
axes[0,1].set_ylabel('CNOT Depth'); axes[0,1].set_title('CNOT Depth vs Grid Size')
axes[1,0].set_ylabel('Total 2Q Gates'); axes[1,0].set_title('Total 2-Qubit Gate Count')
axes[1,1].set_ylabel('Gate Depth / L'); axes[1,1].set_title('Depth Scaling: $O(1)$ vs $O(L)$')

plt.tight_layout()
plt.savefig('scaling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Scaling plots saved to scaling_comparison.png")

## 8. Summary

### Gamma Operator Depth

| Construction | Depth Formula | Ancillas |
|---|---|---|
| Ancilla-based (Jiang et al.) | $7L - 3$ | $L$ |
| Ancilla-free (ours) | $9L + 12$ | **0** |

### End-to-End Fermionic Permutation

| Metric | Baseline (1D Snake) | Ancilla $\Gamma$ | Ancilla-Free $\Gamma$ |
|--------|--------------------|--------------------|------------------------|
| **Gate Depth** | $\sim L^2 = O(N)$ | $\sim 20L = O(\sqrt{N})$ | $\sim 24L = O(\sqrt{N})$ |
| **CNOT Depth** | $\sim 2L^2 = O(N)$ | $\sim 20L = O(\sqrt{N})$ | $\sim 24L = O(\sqrt{N})$ |
| **2Q Gates** | $O(N^2)$ | $O(N\sqrt{N})$ | $O(N\sqrt{N})$ |
| **Ancillas** | 0 | $L$ | **0** |

Both methods achieve a quadratic depth speedup over the 1D baseline ($O(\sqrt{N})$ vs $O(N)$).
The ancilla-free construction eliminates ancilla overhead while maintaining the same asymptotic scaling.

**Note on the ancilla-based construction**: The implementation uses abstract (NamedQubit)
ancillas representing the ideal logical circuit. A physical nearest-neighbor implementation
requires SWAP-based ancilla routing and skip-row CZ decomposition, adding $O(1)$ overhead
per column step. The ancilla-free construction avoids this entirely.